# TFM – Primera prueba con Streamlit

**Autora:** Ona Sánchez Núñez

Página web muy sencilla hecha con Streamlit. Permite subir el dataset *Gene expression cancer RNA-Seq* (Fiorini, 2016) y muestra:

- un mensaje confirmando que se ha leído bien,
- el número de muestras y de genes,
- el porcentaje de cada tipo de cáncer.

Los ficheros que hay que subir son `data.csv` y `labels.csv` (están en la carpeta `Dataset` o se pueden descargar de la [web de UCI](https://archive.ics.uci.edu/dataset/401/gene+expression+cancer+rna+seq)).

## 1. Instalar Streamlit

In [ ]:
!pip install -q streamlit==1.50.0

## 2. Crear la app

`%%writefile` guarda el código de la celda en el fichero `app.py`.

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd

st.title("Clasificación de tipos tumorales")
st.write("Sube los dos ficheros del dataset *Gene expression cancer RNA-Seq* (UCI): "
         "`data.csv` (expresión de los genes) y `labels.csv` (tipo de cáncer de cada muestra).")

archivo_datos = st.file_uploader("Fichero de expresión (data.csv)", type="csv")
archivo_etiquetas = st.file_uploader("Fichero de etiquetas (labels.csv)", type="csv")

# Solo se hace algo cuando se han subido los dos ficheros
if archivo_datos is not None and archivo_etiquetas is not None:
    try:
        with st.spinner("Leyendo los datos..."):
            datos = pd.read_csv(archivo_datos, index_col=0)
            etiquetas = pd.read_csv(archivo_etiquetas, index_col=0)
    except Exception:
        st.error("No se han podido leer los ficheros. Comprueba que son CSV separados por comas.")
        st.stop()

    # Comprobaciones básicas
    if "Class" not in etiquetas.columns:
        st.error("El fichero de etiquetas no tiene la columna 'Class'.")
    elif not datos.index.equals(etiquetas.index):
        st.error("Las muestras de los dos ficheros no coinciden.")
    else:
        st.success("¡El dataset se ha leído correctamente!")

        st.write("Número de muestras:", datos.shape[0])
        st.write("Número de genes:", datos.shape[1])

        st.subheader("Porcentaje de cada tipo de cáncer")
        conteo = etiquetas["Class"].value_counts()
        porcentaje = (conteo / conteo.sum() * 100).round(1)
        tabla = pd.DataFrame({"Muestras": conteo, "Porcentaje (%)": porcentaje})
        st.table(tabla)
        st.bar_chart(porcentaje)


## 3. Abrir la página web

Colab no deja abrir la app directamente, así que se usa **localtunnel**, que crea un enlace público temporal.

1. Ejecuta la celda de abajo. Primero sale un número (la IP de Colab): es la **contraseña** que pide localtunnel.
2. Después sale un enlace `https://....loca.lt`. Ábrelo, pega la contraseña y pulsa *Click to Submit*.
3. Sube `data.csv` y `labels.csv`. El fichero `data.csv` pesa unos 200 MB, así que la subida puede tardar unos minutos.

Para parar la app, detén la celda (botón ■).

In [ ]:
# Contraseña para localtunnel
!curl -s https://loca.lt/mytunnelpassword

# Se lanza Streamlit en segundo plano:
#   --server.maxUploadSize 300 -> permite subir ficheros de hasta 300 MB (data.csv pesa ~200 MB)
#   --server.enableXsrfProtection false -> si no, la subida de ficheros falla a través del túnel
!streamlit run app.py --server.maxUploadSize 300 --server.enableXsrfProtection false --server.headless true &> logs.txt &

# Enlace público a la app
!npx --yes localtunnel --port 8501

### Alternativa: ejecutarla en el ordenador

Con el entorno del repositorio instalado, desde la carpeta donde esté `app.py`:

```bash
streamlit run app.py --server.maxUploadSize 300
```

Se abre sola en el navegador (`http://localhost:8501`).